In [1]:
import fitz

In [2]:
pdf_path="../data/suzukif8dworkshopmanual.pdf"

In [3]:
doc=fitz.open(pdf_path)


In [4]:
import os

In [5]:
print(os.getcwd())

c:\Users\palar\PycharmProjects\RAG_Tutorial\notebook


In [6]:
print(len(doc))

644


In [7]:
page=doc[0]
text=page.get_text()
print(text)

IMPORTANT
WARNING/CAUTION/NOTE
Please read this manual and follow its instructions
carefully. To emphasize special information, the words
WARNING, CAUTION and NOTE have special mean-
ings. Pay special attention to the messages high-
lighted by these signal words.
Indicates a potential hazard that could result
in death or injury.
CAUTION:
Indicates a potential hazard that could result
in vehicle damage.
NOTE:
Indicates special information to make mainte-
nance easier or instructions clearer.
This service manual is intended for autho-
rized Maruti dealers and qualified service
mechanics only. Inexperienced mechanics
or mechanics without the proper tools and
equipment may not be able to properly per-
form the services described in this manual.
Improper repair may result in injury to the
mechanic and may render the vehicle un-
safe for the driver and passengers.



In [8]:
for i in range(5):
    page=doc[i]
    text=page.get_text()
    print(f"\nPAGE {i+1}\n")
    print(text[:1000])
    print("="*50)


PAGE 1

IMPORTANT
WARNING/CAUTION/NOTE
Please read this manual and follow its instructions
carefully. To emphasize special information, the words
WARNING, CAUTION and NOTE have special mean-
ings. Pay special attention to the messages high-
lighted by these signal words.
Indicates a potential hazard that could result
in death or injury.
CAUTION:
Indicates a potential hazard that could result
in vehicle damage.
NOTE:
Indicates special information to make mainte-
nance easier or instructions clearer.
This service manual is intended for autho-
rized Maruti dealers and qualified service
mechanics only. Inexperienced mechanics
or mechanics without the proper tools and
equipment may not be able to properly per-
form the services described in this manual.
Improper repair may result in injury to the
mechanic and may render the vehicle un-
safe for the driver and passengers.


PAGE 2

FOREWORD
This manual contains procedures for diagnosis, maintenance, adjustments, minor service operations,
re

In [9]:
chucks=[]
for page_num in range(len(doc)):
    page=doc[page_num]
    text=page.get_text()
    text=text.strip()
    if len(text)>100:
        system=detect_system(text)
        chunk={
            'manufacturer':"maruti",
            'page':page_num+1,
            "system":system,

            'content':text
        }
        chucks.append(chunk)
print(len(chucks))

NameError: name 'detect_system' is not defined

In [ ]:
import json
with open("maruti_chunks.json","w",encoding="utf-8") as f:
    json.dump(
        chucks,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Done")

Done


In [ ]:
def detect_system(text):
    systems = [
        "ENGINE",
        "FUEL SYSTEM",
        "EMISSION CONTROL",
        "ELECTRICAL",
        "IGNITION",
        "BRAKE",
        "SUSPENSION",
        "TRANSMISSION",
        "OBD"
    ]

    upper = text.upper()

    for system in systems:

        if system in upper:
            return system

    return "UNKNOWN"

In [ ]:
import re

text = "MAINTENANCE AND LUBRICATION 0B-1"
HEADER_PATTERN = r'([A-Z\s]+)\s+([0-9A-Z]+)-(\d+)'

match = re.search(
    HEADER_PATTERN,
    text
)

if match:

    print("SYSTEM:", match.group(1))

    print("SECTION:", match.group(2))

    print("SECTION PAGE:", match.group(3))

SYSTEM: MAINTENANCE AND LUBRICATION
SECTION: 0B
SECTION PAGE: 1


In [ ]:
chunks = []

current_system = None

current_topic = None

content_buffer = []

In [ ]:
TOPIC_KEYWORDS = [

    "Inspection",
    "Replacement",
    "Removal",
    "Installation",
    "Diagnosis",
    "Adjustment",
    "Check",
    "Service",
    "Cleaning",
    "Change"
]

In [ ]:
def istopic(line):
    for keyword in keywords:
        if keyword.lower() in line.lower():
         return True
    return False

In [ ]:
IGNORE_LINES = [

    "WARNING",
    "CAUTION",
    "NOTE"
]

def is_system(line):

    line = line.strip()

    if line in IGNORE_LINES:

        return False

    return (
        line.isupper()
        and len(line.split()) <= 5
    )

In [ ]:
import fitz
import json
import re

# ============================================
# LOAD PDF
# ============================================

pdf_path = "../data/suzukif8dworkshopmanual.pdf"

doc = fitz.open(pdf_path)

print("PDF Loaded")
print("Total Pages:", len(doc))


# ============================================
# KNOWN SYSTEMS
# ============================================

KNOWN_SYSTEMS = [

    "ENGINE",

    "FUEL SYSTEM",

    "IGNITION SYSTEM",

    "BRAKE",

    "EMISSION CONTROL SYSTEM",

    "CLUTCH AND TRANSMISSION",

    "FRONT AND REAR SUSPENSION",

    "ROAD TEST",

    "GENERAL INFORMATION",

    "MAINTENANCE AND LUBRICATION",

    "ELECTRICAL SYSTEM",

    "EXHAUST SYSTEM"
]


# ============================================
# SECTION NAME CANDIDATES
# ============================================

KNOWN_SECTIONS = [

    "GENERAL INFORMATION",

    "MAINTENANCE AND LUBRICATION",

    "ENGINE",

    "BRAKE",

    "FUEL SYSTEM",

    "IGNITION SYSTEM"
]


# ============================================
# TOPIC KEYWORDS
# ============================================

TOPIC_KEYWORDS = [

    "Inspection",
    "Replacement",
    "Removal",
    "Installation",
    "Diagnosis",
    "Adjustment",
    "Check",
    "Service",
    "Cleaning",
    "Change",
    "Testing",
    "Overhaul",
    "Disassembly",
    "Assembly"
]


# ============================================
# REGEX
# ============================================

SECTION_PAGE_PATTERN = r'([0-9A-Z]+)-(\d+)'


# ============================================
# HELPER FUNCTIONS
# ============================================
def is_topic(line):

    if not line:
        return False

    # Reject ALL CAPS
    if line.isupper():
        return False

    # Reject long sentences
    if len(line.split()) > 10:
        return False

    # Must start uppercase
    if not line[0].isupper():
        return False

    # Reject punctuation endings
    if line.endswith("."):
        return False

    # Must contain procedural keywords
    for keyword in TOPIC_KEYWORDS:

        if keyword.lower() in line.lower():

            return True

    return False


# ============================================
# PARSER STATE
# ============================================

chunks = []

current_section = None

current_section_name = None

current_system = None

current_topic = None

content_buffer = []

current_pages = []


# ============================================
# MAIN PARSER LOOP
# ============================================

for page_num in range(len(doc)):

    page = doc[page_num]

    text = page.get_text()

    lines = text.split("\n")

    # ========================================
    # DETECT SECTION FROM PAGE HEADER
    # ========================================

    first_lines = "\n".join(lines[:8])

    section_match = re.search(
        SECTION_PAGE_PATTERN,
        first_lines
    )

    if section_match:

        current_section = section_match.group(1)

    # ========================================
    # PROCESS EACH LINE
    # ========================================

    for line in lines:

        line = line.strip()

        if not line:
            continue


        # ====================================
        # DETECT SECTION NAME
        # ====================================

        if line in KNOWN_SECTIONS:

            current_section_name = line

            print(f"\n[SECTION] {current_section_name}")

            continue


        # ====================================
        # DETECT SYSTEM
        # ====================================

        if line in KNOWN_SYSTEMS:

            current_system = line

            print(f"[SYSTEM] {current_system}")

            continue


        # ====================================
        # DETECT TOPIC
        # ====================================

        if is_topic(line):

            # Save previous chunk first
            if current_topic and content_buffer:

                chunk = {

                    "section": current_section,

                    "section_name": current_section_name,

                    "system": current_system,

                    "topic": current_topic,

                    "pages": current_pages,

                    "content": "\n".join(content_buffer)
                }

                chunks.append(chunk)

            # Start new topic
            current_topic = line

            current_pages = [page_num + 1]

            content_buffer = []

            print(f"   [TOPIC] {current_topic}")

            continue


        # ====================================
        # COLLECT CONTENT
        # ====================================

        if current_topic:

            if (page_num + 1) not in current_pages:

                current_pages.append(page_num + 1)

            content_buffer.append(line)


# ============================================
# SAVE FINAL CHUNK
# ============================================

if current_topic and content_buffer:

    chunk = {

        "section": current_section,

        "section_name": current_section_name,

        "system": current_system,

        "topic": current_topic,

        "pages": current_pages,

        "content": "\n".join(content_buffer)
    }

    chunks.append(chunk)


# ============================================
# RESULTS
# ============================================

print("\n===================================")
print("TOTAL CHUNKS CREATED:", len(chunks))
print("===================================")


# ============================================
# PREVIEW CHUNKS
# ============================================

for i in range(min(5, len(chunks))):

    print("\n===================================")

    print("SECTION:", chunks[i]["section"])

    print("SECTION NAME:", chunks[i]["section_name"])

    print("SYSTEM:", chunks[i]["system"])

    print("TOPIC:", chunks[i]["topic"])

    print("PAGES:", chunks[i]["pages"])

    print("\nCONTENT PREVIEW:\n")

    print(chunks[i]["content"][:700])

    print("\n===================================")


# ============================================
# SAVE JSON
# ============================================

output_file = "hierarchical_chunks.json"

with open(output_file, "w", encoding="utf-8") as f:

    json.dump(
        chunks,
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"\nSaved chunks to: {output_file}")

PDF Loaded
Total Pages: 644
   [TOPIC] This service manual is intended for autho-

[SECTION] GENERAL INFORMATION

[SECTION] ENGINE
   [TOPIC] Engine Diagnosis
[SYSTEM] ELECTRICAL SYSTEM

[SECTION] GENERAL INFORMATION

[SECTION] GENERAL INFORMATION
   [TOPIC] Precautions for Electrical Circuit Service
   [TOPIC] Electrical Circuit Inspection Procedure
   [TOPIC] TOOLS, REQUIRED SERVICE MATERIALS and TIGHT-

[SECTION] GENERAL INFORMATION

[SECTION] GENERAL INFORMATION
   [TOPIC] Such terminal voltage check at low battery voltage will lead
   [TOPIC] While there are various electrical circuit inspection methods, de-
   [TOPIC] When checking system circuits including an electronic control unit

[SECTION] GENERAL INFORMATION
   [TOPIC] Check each terminal visually for poor contact (possibly caused
   [TOPIC] Check contact tension by
   [TOPIC] Continuity check
   [TOPIC] Voltage check
   [TOPIC] SHORT CIRCUIT CHECK (wire harness to ground)

[SECTION] GENERAL INFORMATION
   [TOPIC] Check eac

In [ ]:
toc=doc.get_toc()
pages_for_toc=[]
for item in toc[:20]:
    pages_for_toc.append(item[2])




In [ ]:
toc=toc[4:]
toc

[[1, 'Section 0A ', 4],
 [1, 'Section 0B ', 20],
 [1, 'Section 1A ', 40],
 [1, 'Section 1B ', 52],
 [1, 'Section 3 ', 89],
 [1, 'Section 3A ', 97],
 [1, 'Section 3B ', 101],
 [1, 'Section 3C ', 117],
 [1, 'Section 3D ', 129],
 [1, 'Section 3E ', 150],
 [1, 'Section 3E1 ', 165],
 [1, 'Section 3F ', 182],
 [1, 'Section 4 ', 189],
 [1, 'Section 5 ', 197],
 [1, 'Section 6 ', 234],
 [1, 'Section 6A ', 249],
 [1, 'Section 6A1 ', 304],
 [1, 'Section 6B ', 369],
 [1, 'Section 6C ', 383],
 [1, 'Section 6E1 ', 391],
 [1, 'Section 6F ', 470],
 [1, 'Section 6G ', 485],
 [1, 'Section 6H ', 498],
 [1, 'Section 6K  ', 520],
 [1, 'Section 7A ', 524],
 [1, 'Section 7A1 ', 550],
 [1, 'Section 7C ', 585],
 [1, 'Section 8 ', 599],
 [1, 'Section 9 ', 618],
 [1, 'Section 10 ', 639]]

In [ ]:
sections=[]
for item in range(len(toc)-1):
    sections.append({
        "section":toc[item][1],
        "Start":toc[item][2],
        "end":toc[item+1][2]-1
    })
sections

[{'section': 'Section 0A ', 'Start': 4, 'end': 19},
 {'section': 'Section 0B ', 'Start': 20, 'end': 39},
 {'section': 'Section 1A ', 'Start': 40, 'end': 51},
 {'section': 'Section 1B ', 'Start': 52, 'end': 88},
 {'section': 'Section 3 ', 'Start': 89, 'end': 96},
 {'section': 'Section 3A ', 'Start': 97, 'end': 100},
 {'section': 'Section 3B ', 'Start': 101, 'end': 116},
 {'section': 'Section 3C ', 'Start': 117, 'end': 128},
 {'section': 'Section 3D ', 'Start': 129, 'end': 149},
 {'section': 'Section 3E ', 'Start': 150, 'end': 164},
 {'section': 'Section 3E1 ', 'Start': 165, 'end': 181},
 {'section': 'Section 3F ', 'Start': 182, 'end': 188},
 {'section': 'Section 4 ', 'Start': 189, 'end': 196},
 {'section': 'Section 5 ', 'Start': 197, 'end': 233},
 {'section': 'Section 6 ', 'Start': 234, 'end': 248},
 {'section': 'Section 6A ', 'Start': 249, 'end': 303},
 {'section': 'Section 6A1 ', 'Start': 304, 'end': 368},
 {'section': 'Section 6B ', 'Start': 369, 'end': 382},
 {'section': 'Section 6C

In [ ]:
section_documents = []

for sec in sections:

    text = ""

    for page_num in range(
        sec["Start"] - 1,
        sec["end"]
    ):

        text += doc[page_num].get_text()

    section_documents.append({
        "section": sec["section"],
        "start": sec["Start"],
        "end": sec["end"],
        "text": text
    })

In [ ]:
section_documents[1]["text"][:5000]

'0B\nMAINTENANCE AND LUBRICATION\n0B-1\nSECTION 0B\nMAINTENANCE AND LUBRICATION\nCONTENTS\nPERIODIC MAINTENANCE SCHEDULE \n0B- 2\n. .. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . \nMAINTENANCE SERVICE \n0B- 5\n. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . \nEngine \n0B- 5\n. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . \nIgnition System \n0B-10\n. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . \nFuel System \n0B-11\n. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . \nEmis

In [11]:
!pip install langchain-experimental
!pip install langchain-community
!pip install sentence-transformers
!pip install langchain-huggingface

Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ------------------- -------------------- 262.1/548.1 kB ? eta -:--:--
   ------------------- -------------------- 262.1/548.1 kB ? eta -:--:--
   -------------------------------------- 548.1/548.1 kB 831.3 kB/s eta 0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   -------------------- ------------------- 0.5/1.0 MB 578.7 kB/s eta 0:00:01
   -------------------- ------------------- 0.5/1.0 MB 578.7 kB/s eta 0:00:01
   ------------------------------ --------- 0.8/1.0 MB 645.7 kB/

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 7.34.0 which is incompatible.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.
tensorflow-intel 2.16.1 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.4 which is incompatible.
tensorflow-intel 2.16.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 7.34.0 which is incompatible.


Using cached langchain_huggingface-1.2.2-py3-none-any.whl (31 kB)


In [13]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

C:\Users\palar\AppData\Local\Temp\ipykernel_9540\3215615320.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [14]:
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]